## Data Preprocessing Notebook ##
**Project**: Artificial Intelligence-Based Cell Survival Colony Counting \
**Author**: Mona Wang \
**Supervisors**: Laya Rafiee Sevyeri, Shirin A. Enger 

This Jupyter notebook will:
1. Crop and mask the background of the images in the "preprocess_reprocess" folder
1. Output the results into the preprocess folder

Note that the constants can be changed.

In [14]:
## Constants ##
MINRADIUS = 900 # Note that these radii are for 3024 x 4032 images. The wells take up approx 1/2 of the images
MAXRADIUS = 1000

In [1]:
## Imports ##
import numpy as np
import re

# Image Processing
import cv2 as cv

## Fix the issue of Error #15 "Initializing libiomp5md.dll, but found mk2iomp5md.dll already initialized." ##
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"


## Function for Numerical Ordering ##
numbers = re.compile(r'(\d+)')
def numericalSort(value):
    parts = numbers.split(value)
    parts[1::2] = map(int, parts[1::2])
    return parts

#### Hugh Transform Background Filtering ####

In [2]:
## Paths ##
image_dir = os.path.join('.', 'Data', 'HCT116_Dataset', 'img_raw') # input directory
preprocess_path = os.path.join('.', 'preprocessing') # directory for the intermediaries

In [15]:
# Crop circle with OpenCV
# Documentation: https://docs.opencv.org/3.4/dd/d1a/group__imgproc__feature.html#ga47849c3be0d0406ad3ca45db65a25d2d

dir = sorted(os.listdir(os.path.join('.', 'preprocessing_reprocess')), key=numericalSort) # directory for the intermediaries

for file in dir:
    print("now reprocessing:", file)
    img = cv.imread(os.path.join(image_dir, file))
    img_g = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
    img_g = cv.blur(img_g,(3,3)) #blurring the image


    ## Create blank mask for cropping ##
    height, width, garbage= img.shape
    mask = np.zeros((height,width), np.uint8)
    rows = img_g.shape[0]


    ## Detect circular well in the image ##
    circles = cv.HoughCircles(img_g, cv.HOUGH_GRADIENT, 1, rows / 8,
                            param1=100, param2=40, minRadius=MINRADIUS, maxRadius = MAXRADIUS)
    for i in circles[0, :]:
        center = (int(i[0]), int(i[1]))
        # circle center
        cv.circle(img_g, center, 1, (0, 100, 100), 3)
        # circle outline
        radius = int(i[2])
        cv.circle(img_g, center, radius, (255, 0, 255), 3)
        # Draw on mask
        circle_img = cv.circle(mask,(int(i[0]),int(i[1])),int(i[2]),(255,255,255),thickness=-1)


    ## Overlaying image with mask ##
    masked_data = cv.bitwise_and(img, img, mask=mask)
    _,threshold = cv.threshold(mask,1,255,cv.THRESH_BINARY)


    ## Find rectangular contour and crop the mask ##
    contours = cv.findContours(threshold,cv.RETR_EXTERNAL,cv.CHAIN_APPROX_SIMPLE)
    x,y,w,h = cv.boundingRect(contours[0][0])
    crop = masked_data[y:y+h,x:x+w]
    
    ## Save Intermediaries ##
    cv.imwrite(os.path.join(preprocess_path, file), crop)
    os.remove(os.path.join('.', 'preprocessing_reprocess', file))

now reprocessing: Sample_19-11.jpg
